In [ ]:
from pipeline_functions import *
import torch

In [ ]:
file_path = f"C:/Users/crims/Desktop/Senior Design Code/datasets/Won2022_BIDS/.mat_files/s2.mat"
# preprocessing
X, y = fn_preprocess.preprocess_testing(file_path, use_training=True)

# feature extraction
X_fe, y_fe = fn_feature_extraction.extractFeatures(X, y, k=5, factor=5)

X_encoded = None #spike encoding to implement

# create module
snn = SNNModule.createSNN(8, [256, 128], betas=[0.95, 0.95, 0.95], thresholds=[1, 1, 1])

# load in weights
weights = torch.load('snn_weights.pth', weights_only=True)
snn.load_state_dict(weights)

snn.eval()

# separate X into individual character signals instead of one big dataset
for i in range(len(X)/12/15):
    start_idx = i*12*(15/5)
    end_idx = start_idx + 12*(15/5)
    X_char = X_fe[start_idx:end_idx]
    y_char = y_fe[start_idx:end_idx]

    #get results of model
    with torch.no_grad():
        results = snn(X_char, batch_first=True)

    # select character based on results
    letter, row_idx, col_idx, row_total, col_total = Characterselection.p300_speller_cycle(results)

    print(letter)